# 📖 Notebook 3: Stories & Ephemeral Content

Instagram Stories are photos or videos that **disappear after 24 hours**.  
Over 500 million people use Stories daily — it's one of Instagram's most popular features.

From a system design perspective, Stories are fascinating because they introduce  
**time-based expiration** — data that needs to be automatically cleaned up.

## Learning Objectives

By the end of this notebook, you'll understand:
- How Stories differ from regular posts (data model, lifecycle)
- Using **Redis TTL** (Time To Live) for automatic expiration
- The **story ring** — efficiently querying "which people I follow have active stories?"
- View tracking — knowing who watched your story
- Storage optimization for ephemeral content

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/instagram
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `instagram_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import time
import json
from datetime import datetime, timedelta

# ── Connections ───────────────────────────────────────────
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "instagram_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM stories")
    print(f"✅ PostgreSQL — {cur.fetchone()[0]} stories in database")
    cur.execute("SELECT COUNT(*) FROM stories WHERE expires_at > NOW()")
    print(f"   {cur.fetchone()[0]} active (not expired) stories")
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis()
    r.ping()
    print(f"✅ Redis connected")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 Stories vs Posts: What's Different?

| Feature | Regular Post | Story |
|---------|-------------|-------|
| **Lifetime** | Permanent | 24 hours |
| **Appears in** | Feed (scrolling) | Story ring (top of app) |
| **Interactions** | Likes, comments | Views, replies (DM) |
| **Storage** | Keep forever | Delete after expiry |
| **Multiple per day** | Unusual | Very common (5-10+) |

The key design challenge: **automatic expiration**.  
We need stories to disappear exactly 24 hours after creation — no manual cleanup.

```
Story lifecycle:

Created ──────────────── 24 hours ──────────────── Expired
  │                                                    │
  ├─ Visible in story ring                             ├─ Gone from story ring
  ├─ Viewers can see it                                ├─ Media can be deleted
  └─ Views are tracked                                 └─ View data archived
```

## ❌ Bad Practice: Cron Job Polling the Database

Before we see the good way, let's look at a naive approach many beginners try:  
**a cron job that polls the database every minute looking for expired stories.**

```python
# BAD — runs every 60 seconds, scans the whole stories table
def cleanup_cron_job():
    cur.execute("DELETE FROM stories WHERE expires_at < NOW()")
    # and delete from S3, and remove from any cache, and ...
```

Why this is bad:

| Problem | Why It Hurts |
|---------|-------------|
| **Polling wastes CPU** | Most runs find nothing to delete — still hits the DB |
| **Expiry is imprecise** | A story created at 12:00:30 might linger until 12:01:00 |
| **Reads must re-check time** | Every `SELECT` needs `WHERE expires_at > NOW()` |
| **Hot table writes** | DELETE on a busy table causes index churn and vacuum pressure |
| **Single point of failure** | If the cron skips a run, old stories leak |

Let's run the bad version once to feel how clunky it is, then switch to Redis TTL.

In [ ]:
# ❌ Bad practice: scan the DB for expired rows
conn = get_db()
cur = conn.cursor()
start = time.time()
cur.execute("SELECT COUNT(*) FROM stories WHERE expires_at < NOW()")
expired = cur.fetchone()[0]
elapsed = (time.time() - start) * 1000
conn.close()

print(f"🐢 Cron-style scan found {expired} expired stories in {elapsed:.1f}ms")
print("   Imagine running this every minute across billions of rows...")
print("   → A better approach: let Redis TTL delete data automatically.")

## ✅ Good Practice: Creating a Story with Redis TTL

When a user posts a Story, we:
1. Save the story metadata to PostgreSQL (with an `expires_at` timestamp — for analytics/audit)
2. Cache the story in Redis with a **TTL** (Time To Live) that matches the 24-hour window
3. Add the user to the "active stories" set in Redis

Redis TTL is perfect here — Redis will **automatically delete** the key after the TTL expires.  
No cron jobs, no cleanup scripts, no manual deletion needed.

One honest caveat: Redis expiry is **lazy plus sampled**, not a per-key timer.
A key becomes *unreadable* the instant its TTL passes (every read checks), but the
memory is only reclaimed when something touches the key or the background sampler
happens to pick it. So reads are exact; `INFO memory` lags. That is fine for
Stories and worth knowing before someone asks you in an interview.

In [ ]:
STORY_TTL_SECONDS = 24 * 60 * 60  # 24 hours = 86400 seconds

def create_story(author_id: int, media_key: str) -> dict:
    """
    Create a new story that expires in 24 hours.
    - Saves to PostgreSQL (permanent record for analytics)
    - Caches in Redis with TTL (for fast serving during the 24-hour window)
    """
    conn = get_db()
    cur = conn.cursor()
    r = get_redis()

    # Compute expires_at in SQL, not in Python. `datetime.now()` is the
    # notebook host's wall clock; `created_at` defaults to the database's.
    # Mixing the two makes every story's real lifetime off by the clock skew
    # between them — which on a laptop talking to a UTC container is hours.
    cur.execute(
        """INSERT INTO stories (author_id, media_key, expires_at)
           VALUES (%s, %s, NOW() + (%s || ' seconds')::interval)
           RETURNING id, created_at, expires_at""",
        (author_id, media_key, STORY_TTL_SECONDS)
    )
    story_id, created_at, expires_at = cur.fetchone()
    conn.commit()
    conn.close()

    # Cache in Redis with TTL
    story_data = json.dumps({
        "id": story_id,
        "author_id": author_id,
        "media_key": media_key,
        "created_at": str(created_at),
        "expires_at": str(expires_at)
    })

    # Key: story:{story_id} — auto-expires after 24 hours
    r.setex(f"story:{story_id}", STORY_TTL_SECONDS, story_data)

    # Add story_id to the user's active stories list
    # Key: user_stories:{author_id} — sorted set with creation timestamp as score
    r.zadd(f"user_stories:{author_id}", {str(story_id): time.time()})

    # Mark this user as having active stories
    # Key: active_story_users — set of user IDs with active stories
    r.sadd("active_story_users", str(author_id))

    ttl = r.ttl(f"story:{story_id}")
    print(f"📸 Story #{story_id} created by user {author_id}")
    print(f"   Media: {media_key}")
    print(f"   Expires at: {expires_at}")
    print(f"   Redis TTL: {ttl} seconds ({ttl // 3600}h {(ttl % 3600) // 60}m)")

    return {"story_id": story_id, "author_id": author_id, "ttl": ttl}

# Create a few stories
story1 = create_story(1, "stories/user_1/morning_coffee.jpg")
print()
story2 = create_story(1, "stories/user_1/lunch_selfie.jpg")
print()
story3 = create_story(5, "stories/user_5/sunset_view.jpg")

# ── Redis TTL and the database's expires_at must agree ──────────────────
# If they drift apart, the story vanishes from the cache while the DB still
# thinks it is live (or the other way round) — a bug you only notice hours later.
conn = get_db()
cur = conn.cursor()
cur.execute("""SELECT extract(epoch from (expires_at - NOW()))
               FROM stories WHERE id = %s""", (story1["story_id"],))
db_seconds_left = float(cur.fetchone()[0])
conn.close()

assert story1["ttl"] > 0, "Redis TTL was not set on the story key"
assert abs(story1["ttl"] - STORY_TTL_SECONDS) <= 5, (
    f"Redis TTL is {story1['ttl']}s, expected ~{STORY_TTL_SECONDS}s")
assert abs(db_seconds_left - story1["ttl"]) <= 5, (
    f"database says {db_seconds_left:.0f}s left, Redis says {story1['ttl']}s — "
    f"the two stores disagree about when this story dies")
print(f"\n✅ Redis TTL ({story1['ttl']}s) matches the database's expires_at "
      f"({db_seconds_left:.0f}s left)")

## ⏰ Redis TTL: Automatic Expiration

Let's see TTL in action. We'll create a short-lived story (30 seconds) and watch it expire.

In [ ]:
# Create a story with a very short TTL so we can watch the whole lifecycle.
r = get_redis()

DEMO_TTL_SECONDS = 3
demo_key = "story:demo_ttl"
r.setex(demo_key, DEMO_TTL_SECONDS,
        json.dumps({"content": f"This story disappears in {DEMO_TTL_SECONDS} seconds!"}))

print(f"Created a demo story with a {DEMO_TTL_SECONDS}-second TTL")
print(f"   → Open RedisInsight (http://localhost:5540) and search for key '{demo_key}'")
print(f"   → Watch the TTL count down!\n")

assert r.exists(demo_key), "the story must be readable immediately after creation"

# Poll until Redis removes it. Bounded, so a broken Redis fails fast instead of
# hanging, and generous enough that a slow machine is not a flaky failure.
start_poll = time.time()
expired_after = None
while time.time() - start_poll < DEMO_TTL_SECONDS + 10:
    elapsed = time.time() - start_poll
    if r.exists(demo_key):
        print(f"   ⏱️  T+{elapsed:4.1f}s: TTL = {r.ttl(demo_key)}s — story exists ✅")
        time.sleep(1)
    else:
        expired_after = elapsed
        print(f"   ⏱️  T+{elapsed:4.1f}s: story EXPIRED ❌ — Redis deleted it by itself")
        break

# The entire lesson is that Redis does the deletion for us. If the key is still
# alive well past its TTL, this notebook is teaching something that is not true.
assert expired_after is not None, (
    f"'{demo_key}' was still alive {DEMO_TTL_SECONDS + 10}s after a "
    f"{DEMO_TTL_SECONDS}s TTL — TTL expiry is not working on this Redis")
assert r.ttl(demo_key) == -2, (
    f"an expired key should report TTL -2 (gone), got {r.ttl(demo_key)}")

print(f"\n💡 Gone after {expired_after:.1f}s with zero cleanup code.")
print("   This is exactly how Instagram Stories expire — except the TTL is 86400s.")

## 💍 The Story Ring

At the top of the Instagram app, you see a row of circles (the "story ring").  
Each circle represents a user you follow who has active (non-expired) stories.

```
┌──────────────────────────────────────────────────────────┐
│  (You)   (Alice)  (Bob)   (User5)  (User10)  ...        │
│   🔴       🔴      🔴      🔴        🔴                  │
│  Your    Alice's  Bob's   User5's  User10's              │
│  Story   Story    Story   Story    Story                 │
└──────────────────────────────────────────────────────────┘
```

To build this, we need to answer: **"Which users I follow have active stories right now?"**  
This query runs every time you open the app — it needs to be fast.

In [ ]:
def load_active_stories_into_redis() -> int:
    """
    `init.sql` seeded stories straight into PostgreSQL, so Redis knows nothing
    about them. In production every story is written through create_story();
    here we replay the seeded rows so the story ring has something to show.

    Each key gets a TTL equal to its REMAINING lifetime — not a fresh 24 hours,
    which would silently resurrect stories that are nearly dead.
    """
    conn = get_db()
    cur = conn.cursor()
    r = get_redis()

    cur.execute("""
        SELECT id, author_id, media_key,
               extract(epoch from created_at)          AS created_ts,
               extract(epoch from (expires_at - NOW())) AS ttl_seconds
        FROM stories
        WHERE expires_at > NOW()
    """)
    rows = cur.fetchall()
    conn.close()

    loaded = 0
    pipe = r.pipeline()
    for story_id, author_id, media_key, created_ts, ttl_seconds in rows:
        ttl = int(ttl_seconds)
        if ttl <= 0:
            continue
        pipe.setex(f"story:{story_id}", ttl, json.dumps({
            "id": story_id, "author_id": author_id, "media_key": media_key,
        }))
        pipe.zadd(f"user_stories:{author_id}", {str(story_id): float(created_ts)})
        pipe.sadd("active_story_users", str(author_id))
        loaded += 1
    pipe.execute()
    return loaded


loaded = load_active_stories_into_redis()

# A long-running container can outlive every seeded story. Make fresh ones
# rather than letting the story-ring section quietly demonstrate nothing.
if loaded == 0:
    print("No unexpired seed stories left — creating fresh ones for the demo.\n")
    for uid in (8, 12, 13):
        create_story(uid, f"stories/user_{uid}/fresh.jpg")
    print()
    loaded = load_active_stories_into_redis()

print(f"✅ {loaded} active stories loaded into Redis")
print(f"   Users with active stories: "
      f"{len(get_redis().smembers('active_story_users'))}")
assert loaded > 0, "there must be at least one active story for the ring to show"

In [ ]:
def get_story_ring(user_id: int) -> list:
    """
    Get the story ring for a user:
    1. Get the list of people this user follows
    2. Intersect with the set of users who have active stories
    3. Return them ordered the way Instagram does: **unseen rings first**,
       then by the recency of each author's newest story

    Everything here is batched. The obvious implementation does one Redis call
    and one SQL query *per followed user*, which is exactly the N+1 pattern
    Notebook 2 spent a whole section getting rid of.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    r = get_redis()
    start = time.time()

    # Step 1: Get followed users
    cur.execute(
        "SELECT followee_id FROM follows WHERE follower_id = %s",
        (user_id,)
    )
    followee_ids = [row["followee_id"] for row in cur.fetchall()]

    # Step 2: Intersect with the "who has active stories" set — one round-trip
    active_users = r.smembers("active_story_users")
    users_with_stories = [uid for uid in followee_ids if str(uid) in active_users]
    if not users_with_stories:
        conn.close()
        print(f"💍 Story ring for user {user_id}: "
              f"nobody they follow has an active story")
        return []

    # Step 3: newest story + story count per author — one pipelined round-trip
    pipe = r.pipeline()
    for uid in users_with_stories:
        pipe.zrevrange(f"user_stories:{uid}", 0, 0, withscores=True)
        pipe.zcard(f"user_stories:{uid}")
    redis_rows = pipe.execute()

    entries = []
    for i, uid in enumerate(users_with_stories):
        newest, count = redis_rows[2 * i], redis_rows[2 * i + 1]
        if newest and count:
            entries.append({
                "user_id": uid,
                "newest_story_id": int(newest[0][0]),
                "newest_ts": newest[0][1],
                "story_count": count,
            })
    if not entries:
        conn.close()
        return []

    # Step 4: has this viewer already watched each author's newest story?
    # One more pipelined round-trip, not one SISMEMBER per author.
    pipe = r.pipeline()
    for e in entries:
        pipe.sismember(f"story_views:{e['newest_story_id']}", str(user_id))
    seen_flags = pipe.execute()

    # Step 5: display names — one SQL query for all of them
    cur.execute(
        "SELECT id, username, display_name FROM users WHERE id = ANY(%s)",
        ([e["user_id"] for e in entries],)
    )
    user_info = {row["id"]: row for row in cur.fetchall()}
    conn.close()

    for e, seen in zip(entries, seen_flags):
        e["has_unseen"] = not seen
        e["username"] = user_info[e["user_id"]]["username"]
        e["display_name"] = user_info[e["user_id"]]["display_name"]

    # Unseen rings first, then newest story first within each group.
    entries.sort(key=lambda e: (not e["has_unseen"], -e["newest_ts"]))

    elapsed = (time.time() - start) * 1000
    print(f"💍 Story ring for user {user_id}:")
    print(f"   Follows {len(followee_ids)} users, {len(entries)} have active stories")
    print(f"   Query time: {elapsed:.1f}ms "
          f"(2 SQL queries + 3 Redis round-trips, regardless of follow count)")

    return entries


# User 10 follows users 8, 12 and 13 (among others). The seed gives every user
# with id <= 20 two stories, so this ring should NOT be empty — an earlier
# version of this notebook demoed a user who follows nobody with stories, and
# the "empty ring" output looked like a data problem rather than a bug.
DEMO_VIEWER = 10

# Clear view marks this demo left behind on a previous run, so "unseen first"
# starts from a known state and the notebook is re-runnable.
r = get_redis()
conn = get_db()
cur = conn.cursor()
cur.execute("SELECT followee_id FROM follows WHERE follower_id = %s", (DEMO_VIEWER,))
for (followee_id,) in cur.fetchall():
    for sid in r.zrange(f"user_stories:{followee_id}", 0, -1):
        r.srem(f"story_views:{sid}", str(DEMO_VIEWER))
conn.close()

ring = get_story_ring(user_id=DEMO_VIEWER)
assert ring, (
    f"story ring for user {DEMO_VIEWER} is empty — "
    "run load_active_stories_into_redis() in the cell above first")
assert all(e["has_unseen"] for e in ring), "nothing has been viewed yet"

print(f"\n   Story Ring:")
for entry in ring:
    print(f"   🔴 {entry['display_name']} (@{entry['username']}) — "
          f"{entry['story_count']} stories, unseen")

# ── The ordering claim in the docstring has to be true, not aspirational ──
watched = ring[0]
r.sadd(f"story_views:{watched['newest_story_id']}", str(DEMO_VIEWER))
ring2 = get_story_ring(user_id=DEMO_VIEWER)

order_key = [(not e["has_unseen"], -e["newest_ts"]) for e in ring2]
assert order_key == sorted(order_key), f"ring is not unseen-first/newest-first: {order_key}"
assert ring2[-1]["user_id"] == watched["user_id"], (
    f"a fully watched ring must sink to the end, got "
    f"{[e['user_id'] for e in ring2]} with {watched['user_id']} watched")
assert not ring2[-1]["has_unseen"], "the watched ring should be marked seen"

print(f"\n   After user {DEMO_VIEWER} watches @{watched['username']}'s newest "
      f"story, that ring drops to the end:")
for entry in ring2:
    marker = "🔴" if entry["has_unseen"] else "⚪"
    print(f"   {marker} @{entry['username']} "
          f"({'unseen' if entry['has_unseen'] else 'seen'})")

## 👁️ View Tracking

Instagram shows you who viewed your story. This requires tracking every view  
without slowing down the story viewing experience.

We use Redis sets for fast, deduplicated view tracking:

In [ ]:
def view_story(story_id: int, viewer_id: int):
    """
    Record that a user viewed a story.
    Uses Redis set for fast deduplication (a user can only view once).
    Also saves to PostgreSQL for permanent analytics.
    """
    r = get_redis()
    conn = get_db()
    cur = conn.cursor()

    # Check if story still exists (not expired)
    story_data = r.get(f"story:{story_id}")
    if not story_data:
        print(f"❌ Story #{story_id} has expired or doesn't exist")
        conn.close()
        return

    # Add to Redis view set (returns 1 if new, 0 if already viewed)
    is_new_view = r.sadd(f"story_views:{story_id}", str(viewer_id))

    # Set TTL on view set to match story expiration
    story_ttl = r.ttl(f"story:{story_id}")
    if story_ttl > 0:
        r.expire(f"story_views:{story_id}", story_ttl)

    if is_new_view:
        # Save to PostgreSQL for permanent records
        cur.execute(
            """INSERT INTO story_views (story_id, viewer_id)
               VALUES (%s, %s)
               ON CONFLICT DO NOTHING""",
            (story_id, viewer_id)
        )
        conn.commit()
        print(f"👁️  User {viewer_id} viewed story #{story_id} (new view)")
    else:
        print(f"👁️  User {viewer_id} already viewed story #{story_id} (duplicate ignored)")

    conn.close()

def get_story_viewers(story_id: int) -> list:
    """Get all viewers of a story (from Redis for speed)."""
    r = get_redis()
    viewers = r.smembers(f"story_views:{story_id}")
    return [int(v) for v in viewers]

# Simulate some views on story1
sid = story1["story_id"]
VIEWERS = [2, 3, 5, 7, 10, 15]
print(f"Simulating views on story #{sid}:\n")

for viewer_id in VIEWERS:
    view_story(sid, viewer_id)

# Try a duplicate view
print()
view_story(sid, 5)  # User 5 views again — should be deduplicated

viewers = sorted(get_story_viewers(sid))
print(f"\n📊 Total unique viewers: {len(viewers)}")
print(f"   Viewer IDs: {viewers}")

# ── Dedup is the claim; assert it in both stores ────────────────────────
assert viewers == sorted(VIEWERS), (
    f"expected exactly {sorted(VIEWERS)} unique viewers in Redis, got {viewers}")

conn = get_db()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM story_views WHERE story_id = %s", (sid,))
db_views = cur.fetchone()[0]
conn.close()
assert db_views == len(VIEWERS), (
    f"PostgreSQL should hold {len(VIEWERS)} view rows, got {db_views} — "
    "the Redis set and the durable record have diverged")

# The Redis view set must not outlive the story it belongs to.
view_ttl, story_ttl = get_redis().ttl(f"story_views:{sid}"), get_redis().ttl(f"story:{sid}")
assert 0 < view_ttl <= story_ttl, (
    f"story_views TTL ({view_ttl}s) must be set and must not exceed the "
    f"story's own TTL ({story_ttl}s) — otherwise view sets leak forever")
print(f"\n✅ {db_views} durable rows, {len(viewers)} Redis members, "
      f"view set expires in {view_ttl}s (story: {story_ttl}s)")

## 🔄 Story Expiration & Cleanup

Redis TTL handles removing the cached story data automatically.  
But we also need to:
1. Remove the user from the "active_story_users" set when all their stories expire
2. Clean up the user_stories sorted set
3. Optionally delete the media from S3/MinIO

In production, this is typically handled by a **background cleanup job**.

Note what is *not* automatic here. `story:{id}` has a TTL, so Redis removes it.
The **index** structures — `user_stories:{uid}` and `active_story_users` — have
no TTL of their own, because members of a set cannot expire individually. They
are what the sweeper exists for. This lab keeps the S3/MinIO objects so the
earlier notebooks stay re-runnable; a real system would delete them here too,
which is where most of the storage saving actually comes from.

In [ ]:
def cleanup_expired_stories() -> int:
    """
    Background job that cleans up expired story data.

    Redis has already deleted the `story:{id}` keys — that part is free. What is
    left behind is the *index*: the per-user sorted set and the active-users set.
    Set and sorted-set members cannot carry their own TTL, so nothing expires
    them. Sweeping them is the whole job.

    In production this runs periodically (e.g. every 5 minutes).
    """
    r = get_redis()
    cleaned = 0

    # Snapshot the active users up front — we mutate this set as we go.
    active_users = r.smembers("active_story_users")

    for user_id_str in active_users:
        story_ids = r.zrange(f"user_stories:{user_id_str}", 0, -1)

        # A story id whose `story:{id}` key is gone has expired.
        expired_ids = [sid for sid in story_ids if not r.exists(f"story:{sid}")]

        if expired_ids:
            r.zrem(f"user_stories:{user_id_str}", *expired_ids)
            cleaned += len(expired_ids)

        # If the user has no more active stories, drop them from the ring index.
        if r.zcard(f"user_stories:{user_id_str}") == 0:
            r.srem("active_story_users", user_id_str)
            print(f"   Removed user {user_id_str} from active_story_users "
                  f"(no stories left)")

    print(f"🧹 Cleanup complete: {cleaned} expired story references removed")
    return cleaned


# Give the sweeper something to actually sweep. Without this the job runs, finds
# nothing, prints "0 removed" and proves precisely nothing.
r = get_redis()
GHOST_USER, GHOST_STORY = 999001, 999002
r.zadd(f"user_stories:{GHOST_USER}", {str(GHOST_STORY): time.time()})
r.sadd("active_story_users", str(GHOST_USER))
assert not r.exists(f"story:{GHOST_STORY}"), (
    "the ghost story's data key must already be absent — that is the state "
    "Redis leaves behind after a TTL fires")

print(f"Planted a dangling index entry: user_stories:{GHOST_USER} → "
      f"story {GHOST_STORY} (whose story:{GHOST_STORY} key is already gone)\n")

cleaned = cleanup_expired_stories()

assert cleaned >= 1, (
    f"the sweeper should have removed the dangling index entry, got {cleaned}")
assert r.zcard(f"user_stories:{GHOST_USER}") == 0, (
    "the dangling sorted-set entry survived cleanup")
assert not r.sismember("active_story_users", str(GHOST_USER)), (
    "a user with no stories left must be dropped from active_story_users, "
    "or the story ring keeps checking them forever")

# Live stories must survive the sweep — a cleanup job that deletes real data
# is worse than one that leaks.
assert r.smembers("active_story_users"), "the sweep removed every active user"
assert r.exists(f"story:{story1['story_id']}"), (
    f"story #{story1['story_id']} is still within its 24h TTL and must survive cleanup")

print(f"\n✅ Dangling entry swept, empty user dropped, live stories untouched "
      f"({len(r.smembers('active_story_users'))} users still have stories)")

## 📊 Story Architecture Summary

```
┌─────────┐    POST /stories    ┌───────────┐
│ Client  │────────────────────►│  Story    │
│         │                     │  Service  │
└─────────┘                     └─────┬─────┘
                                      │
                        ┌─────────────┼─────────────┐
                        │             │             │
                        ▼             ▼             ▼
                  ┌──────────┐  ┌──────────┐  ┌──────────┐
                  │PostgreSQL│  │  Redis   │  │  MinIO   │
                  │          │  │          │  │  (S3)    │
                  │ stories  │  │ story:ID │  │  media   │
                  │ table    │  │ (TTL 24h)│  │  bytes   │
                  │ (perm.)  │  │          │  │          │
                  └──────────┘  │ user_    │  └──────────┘
                                │ stories: │
                                │ {uid}    │
                                │          │
                                │ active_  │
                                │ story_   │
                                │ users    │
                                └──────────┘
```

### Redis Keys Used

| Key Pattern | Type | TTL | Purpose |
|-------------|------|-----|----------|
| `story:{id}` | String (JSON) | 24h | Story data cache |
| `user_stories:{uid}` | Sorted Set | ∞ (cleaned up) | User's active story IDs |
| `story_views:{id}` | Set | Same as story | Viewer IDs for a story |
| `active_story_users` | Set | ∞ (cleaned up) | Users with active stories |

## 🧠 Key Takeaways

1. **Redis TTL** is perfect for ephemeral content — set it once, Redis handles deletion
2. **Dual storage**: PostgreSQL for permanent records/analytics, Redis for fast serving
3. **Story ring** query needs to be fast — use Redis sets to intersect "who I follow" with "who has stories"
4. **View tracking** uses Redis sets for deduplication (SADD returns 0 if already a member)
5. **Background cleanup** handles cascading deletes (user_stories set, active_story_users)

### Interview Tips

- Mention TTL immediately when discussing ephemeral content
- Explain the dual-storage pattern: Redis for serving (fast), DB for analytics (permanent)
- Discuss what happens when Redis goes down (fall back to DB query with `WHERE expires_at > NOW()`)
- Mention storage optimization: delete media from S3 after expiry to save costs

### What This Toy Version Does NOT Do

- **No S3 cleanup.** The media objects outlive the story here; in production
  deleting them is where the storage saving comes from
- **No Redis-down fallback.** `get_story_ring` reads Redis only. A real service
  falls back to `SELECT ... WHERE expires_at > NOW()` and takes the latency hit
- **`active_story_users` is one global key.** At Instagram scale that is a hot
  key; production shards it or checks per-followee keys directly
- **View counts are per-story sets.** For a celebrity with 50M viewers a Redis
  set is the wrong shape — you would use HyperLogLog for the count and keep only
  a recent sample of viewer ids

### What's Next?

In **Notebook 4**, we'll build Instagram's **Explore page** — a recommendation system  
that suggests posts from accounts you don't follow based on your engagement patterns.